# Notebook 02: Splits and downsampling

Builds train, validation, holdout, and test split manifests as CSV files.

No image binaries are modified or copied.

Per seed:
- Stratified 30-image Togunwa holdout + 160-image working set split into 5 CV folds.
- Balanced 1:1 Kermany downsamples at sizes 50, 100, and 190.

Downsamples are strictly nested (50 ⊂ 100 ⊂ 190) to ensure scaling analysis adds data rather than resampling.


## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
from collections import Counter
import pandas as pd
from PIL import Image

import config
from src import splits as SP

importlib.reload(config)
importlib.reload(SP)

config.ensure_output_dirs()
config.summary()


Mounted at /content/drive
Cross-Population CXR Asymmetry Experiment Configuration
-------------------------------------------------------
DATA_ROOT           : /content/drive/MyDrive/MSc AI DISSERTATION/data
Architectures       : ['mobilenet_v2', 'efficientnet_b0']
Directions          : ['K2N', 'N2K']
Matched sizes       : [50, 100, 190]
Class balance       : 1:1 Ratio
Total training runs : 12 (2 archs x 2 dirs x 3 sizes)
CV folds            : 5 | Holdout: 30
Seeds               : [42, 43, 44]


In [ ]:
MANIFEST_DIR = config.RESULTS_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print("Manifest destination ->", MANIFEST_DIR)


Manifest destination -> /content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry/outputs/results/manifests


## 2. Collect Source Records

In [ ]:
togunwa_records = SP.collect_records(config.TOGUNWA_CLASSES, config.CLASS_TO_IDX)
kermany_train_records = SP.collect_records(config.KERMANY_SPLITS["train"], config.CLASS_TO_IDX)
kermany_test_records = SP.collect_records(config.KERMANY_SPLITS["test"], config.CLASS_TO_IDX)

def count_classes(records):
    return dict(Counter(r["class"] for r in records))

print("Togunwa       :", count_classes(togunwa_records), "total", len(togunwa_records))
print("Kermany train :", count_classes(kermany_train_records), "total", len(kermany_train_records))
print("Kermany test  :", count_classes(kermany_test_records), "total", len(kermany_test_records))


Togunwa       : {'NORMAL': 93, 'PNEUMONIA': 97} total 190
Kermany train : {'NORMAL': 1349, 'PNEUMONIA': 3883} total 5232
Kermany test  : {'NORMAL': 234, 'PNEUMONIA': 390} total 624


## 3. Record Original Colour Modes

In [ ]:
def build_mode_lookup(records):
    lookup = {}
    for r in records:
        try:
            with Image.open(r["path"]) as im:
                lookup[r["path"]] = im.mode
        except Exception:
            lookup[r["path"]] = "unreadable"
    return lookup

all_records = togunwa_records + kermany_train_records + kermany_test_records
mode_lookup = build_mode_lookup(all_records)

print("\nColor mode distribution:")
for mode, count in Counter(mode_lookup.values()).most_common():
    print(f"  {mode:10s} {count}")



Color mode distribution:
  L          5573
  RGB        431
  RGBA       42


## 4. Build Splits Across All Seeds

In [ ]:
manifest_frames = []

def add_manifest(records, split, dataset, direction, size, fold, seed):
    df = SP.records_to_manifest(records, split, dataset, direction, size, fold, seed)
    df = SP.attach_modes(df, mode_lookup)
    manifest_frames.append(df)

for seed in config.SEEDS:
    # Togunwa: 30-image holdout and 160-image working set
    holdout, working = SP.stratified_holdout(togunwa_records, config.TOGUNWA_HOLDOUT_SIZE, seed)
    add_manifest(holdout, "holdout", "togunwa", "K2N", "", "", seed)

    # 5-Fold Cross Validation on working set
    for f, (tr, val) in enumerate(SP.kfold_splits(working, config.N_FOLDS, seed)):
        add_manifest(tr, "train", "togunwa", "N2K", 190, f, seed)
        add_manifest(val, "val", "togunwa", "N2K", 190, f, seed)

    # Kermany: Nested downsampling (sample max size 190 first, then slice 50 ⊂ 100 ⊂ 190)
    ds_max = SP.balanced_downsample(kermany_train_records, max(config.MATCHED_SIZES), config.CLASS_TO_IDX, seed)
    by_cls = SP.group_by_class(ds_max)

    for size in config.MATCHED_SIZES:
        per_cls = size // 2
        subset = []
        for cls in sorted(by_cls.keys()):
            subset.extend(by_cls[cls][:per_cls])
        add_manifest(subset, "train", "kermany", "K2N", size, "", seed)

    # Kermany evaluation test set (appended once)
    if seed == config.SEEDS[0]:
        add_manifest(kermany_test_records, "test", "kermany", "N2K", "", "", "all")

print(f"\nCreated {len(manifest_frames)} manifest fragments.")



Created 43 manifest fragments.


## 5. Combine, Verify Invariants, and Export

In [ ]:
manifest = pd.concat(manifest_frames, ignore_index=True)
print("Combined manifest shape:", manifest.shape)
print("\nManifest record breakdown:")
print(manifest.groupby(["dataset", "direction", "split"]).size().to_string())


Combined manifest shape: (4134, 10)

Manifest record breakdown:
dataset  direction  split  
kermany  K2N        train      1020
         N2K        test        624
togunwa  K2N        holdout      90
         N2K        train      1920
                    val         480


In [ ]:
problems = []
for seed in config.SEEDS:
    s = manifest[manifest["seed"].astype(str) == str(seed)]

    # Holdout / Working isolation check
    ho_paths = set(s[(s.dataset == "togunwa") & (s.split == "holdout")]["path"])
    tt_paths = set(s[(s.dataset == "togunwa") & (s.split == "train")]["path"])
    tv_paths = set(s[(s.dataset == "togunwa") & (s.split == "val")]["path"])
    if ho_paths & (tt_paths | tv_paths):
        problems.append(f"Seed {seed}: Holdout overlaps with working set")

    # CV Fold partition check
    val_by_fold = s[(s.dataset == "togunwa") & (s.split == "val")].groupby("fold")["path"].apply(set)
    all_val = set().union(*val_by_fold) if len(val_by_fold) else set()
    work0 = set(s[(s.dataset == "togunwa") & (s.split.isin(["train", "val"])) & (s.fold == 0)]["path"])
    if len(val_by_fold) and all_val != work0:
        problems.append(f"Seed {seed}: Validation folds do not partition working set")

    # Downsample balance and subset nesting check
    sizes = {}
    for size in config.MATCHED_SIZES:
        d = s[(s.dataset == "kermany") & (s.split == "train") & (s["size"] == size)]
        n = (d["class"] == "NORMAL").sum()
        p = (d["class"] == "PNEUMONIA").sum()
        sizes[size] = set(d["path"])
        if n != p or n != size // 2:
            problems.append(f"Seed {seed}: Size {size} imbalanced ({n} NORMAL / {p} PNEUMONIA)")

    if not (sizes.get(50, set()) <= sizes.get(100, set()) <= sizes.get(190, set())):
        problems.append(f"Seed {seed}: Downsamples violate subset nesting (50 ⊄ 100 ⊄ 190)")

print("\n--- INVARIANT AUDIT ---")
if problems:
    print("PROBLEMS DETECTED:")
    for p in problems:
        print(f"  [FAIL] {p}")
else:
    print("All invariants hold: Holdout isolated, folds partition cleanly, downsamples balanced and nested.")



--- INVARIANT AUDIT ---
All invariants hold: Holdout isolated, folds partition cleanly, downsamples balanced and nested.


In [ ]:
# Write manifest files
manifest.to_csv(MANIFEST_DIR / "splits_all.csv", index=False)
print(f"\nWrote: {MANIFEST_DIR / 'splits_all.csv'} {manifest.shape}")

for seed in config.SEEDS:
    sp = manifest[manifest["seed"].astype(str) == str(seed)]
    sp.to_csv(MANIFEST_DIR / f"splits_seed{seed}.csv", index=False)

manifest[manifest["seed"].astype(str) == "all"].to_csv(MANIFEST_DIR / "eval_targets.csv", index=False)
print("Wrote per-seed and evaluation manifests.")



Wrote: /content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry/outputs/results/manifests/splits_all.csv (4134, 10)
Wrote per-seed and evaluation manifests.


**Next:** notebook 03 preprocesses images and runs the 12 training configs.